**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 40: 프로젝트 학습 — MCP + LangGraph + A2A 통합 에이전트

## 🎯 학습 목표

세션 33 ~ 38 에서 익힌 **Tool Calling · MCP · A2A · LangGraph** 를 한 번에 묶어,
실제로 동작하는 «멀티 도구 에이전트»를 직접 만든다.

```
[Part 5 프로젝트 파이프라인]
   39 데이터 파이프라인  →  40 에이전트 구축 (이번 노트북)  →  41 평가  →  42 배포
```

### 이번 노트북에서 만드는 것

1. **지식 베이스(KB)** — 39 에서 만든 `train.json` 을 RAG 코퍼스로 인덱싱
2. **MCP 도구 서버** — `search_kb · calculate · get_time` 3개 도구
3. **LangGraph 워크플로** — 라우터 → 전문 에이전트 → 답변 생성
4. **A2A 노출** — `/.well-known/agent.json` + `/a2a` 엔드포인트
5. **AgentPipeline 클래스** — 41 평가 · 42 배포에서 임포트해 그대로 사용

### 산출물

| 파일 | 용도 |
|---|---|
| `agent_kb.pkl`         | TF-IDF 인덱스 (KB) |
| `mcp_server.py`        | FastMCP 도구 서버 (실행 가능) |
| `a2a_server.py`        | A2A 인터페이스 FastAPI (실행 가능) |
| `agent_pipeline.py`    | AgentPipeline 모듈 — 41/42 에서 import |
| `agent_config.json`    | 에이전트 설정 |


---

## 1️⃣ 환경 설정 & 프로젝트 계획 로드


In [ ]:
# 📦 라이브러리 임포트
import os
import json
import pickle
import time
import re
import math
from datetime import datetime
from pathlib import Path
from typing import TypedDict, Annotated, Literal

# 강의용 — 외부 의존성 최소화. LangGraph 가 없으면 자동 폴백.
try:
    from langgraph.graph import StateGraph, END
    HAS_LANGGRAPH = True
except ImportError:
    HAS_LANGGRAPH = False
    print("⚠ langgraph 미설치 — 단순 라우터로 폴백합니다")
    print("   설치: pip install langgraph")

# OpenAI (옵션)
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 프로젝트 계획 로드 (39 에서 만든 것)
PLAN_PATH = "project_plan.json"
if os.path.exists(PLAN_PATH):
    project_plan = json.load(open(PLAN_PATH, encoding="utf-8"))
    print(f"✅ project_plan.json 로드: 도메인={project_plan.get('domain', '?')}")
else:
    project_plan = {"domain": "general", "agent_name": "demo-agent"}
    print("ℹ project_plan.json 없음 — 기본값 사용")

OUTPUT_DIR = "./outputs/agent"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"✅ 산출물 디렉터리: {OUTPUT_DIR}")
print(f"   LangGraph: {'available' if HAS_LANGGRAPH else 'fallback to simple router'}")
print(f"   OpenAI:    {'available' if HAS_OPENAI else 'fallback to template responses'}")


---

## 2️⃣ 지식 베이스(KB) 구축 — 39 데이터를 RAG 코퍼스로

세션 39 에서 만든 `train.json` 의 «질문/답변» 쌍을 RAG 검색용 코퍼스로 인덱싱합니다.
임베딩 대신 강의용으로 **TF-IDF** 를 사용해 외부 API 의존을 없앱니다 (실제 운영은 OpenAI 임베딩 + Chroma 추천).


In [ ]:
# 📚 KB 구축
TRAIN_PATH = "outputs/data/train.json"   # 39 출력

if not os.path.exists(TRAIN_PATH):
    print(f"⚠ {TRAIN_PATH} 없음 — 데모용 KB 생성")
    # 데모 데이터 (39 미수행 시)
    train_data = [
        {"instruction": "MCP가 뭔가요?", "output": "MCP(Model Context Protocol)는 AI 에이전트와 도구 사이의 표준 통신 규약입니다. Anthropic이 2024년 발표했습니다."},
        {"instruction": "A2A는 무엇인가요?", "output": "A2A(Agent-to-Agent)는 독립된 에이전트들이 HTTP/JSON-RPC 로 협업하는 프로토콜입니다. Google 이 주도합니다."},
        {"instruction": "LangGraph의 특징은?", "output": "LangGraph는 상태 머신 기반 에이전트 워크플로 프레임워크입니다. 노드와 엣지로 흐름을 정의합니다."},
        {"instruction": "Tool Calling이란?", "output": "Tool Calling은 LLM이 외부 함수를 호출하도록 하는 기능입니다. OpenAI Function Calling 이 대표적입니다."},
        {"instruction": "Agent Card는 어디에 두나요?", "output": "/.well-known/agent.json 경로에 둡니다. RFC 8615 well-known URI 표준 위치입니다."},
    ]
else:
    train_data = json.load(open(TRAIN_PATH, encoding="utf-8"))
    if isinstance(train_data, dict) and "data" in train_data:
        train_data = train_data["data"]

print(f"KB 문서: {len(train_data)}개")

# 각 항목을 검색용 문장으로 변환
def to_doc(item):
    q = item.get("instruction") or item.get("input") or item.get("question") or ""
    a = item.get("output") or item.get("response") or item.get("answer") or ""
    return f"Q: {q}\nA: {a}"

docs = [to_doc(item) for item in train_data]

# TF-IDF 인덱스
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
doc_matrix = vectorizer.fit_transform(docs)

# 저장
with open(os.path.join(OUTPUT_DIR, "agent_kb.pkl"), "wb") as f:
    pickle.dump({"docs": docs, "vectorizer": vectorizer, "doc_matrix": doc_matrix}, f)

print(f"✅ TF-IDF 인덱스 구축 완료 — {doc_matrix.shape[0]}개 문서, {doc_matrix.shape[1]}개 피처")


def search_kb(query: str, top_k: int = 3) -> list[dict]:
    """KB 검색 — query 와 유사한 문서 Top-K 반환."""
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix).ravel()
    top_idx = sims.argsort()[::-1][:top_k]
    return [{"score": float(sims[i]), "doc": docs[i]} for i in top_idx]


# 테스트
print("\n[검색 테스트] '에이전트 프로토콜이 뭐야'")
for hit in search_kb("에이전트 프로토콜이 뭐야", top_k=2):
    print(f"  ({hit['score']:.3f})  {hit['doc'][:80]}...")


---

## 3️⃣ MCP 도구 서버 (Session 36 응용)

세션 36 에서 배운 FastMCP 로 **`search_kb · calculate · get_time`** 3개 도구를 노출합니다.

> **참고**: 이 노트북에서는 `%%writefile` 로 서버 코드를 파일에 저장만 합니다.
> 실제 동작 확인은 별도 터미널에서 `python mcp_server.py` 로 실행 후
> Claude Desktop · Cursor 같은 MCP Host 에서 연결하거나, 이어지는 §4 의 클라이언트로 호출합니다.


In [ ]:
%%writefile mcp_server.py
"""MCP 도구 서버 — search_kb / calculate / get_time (Session 36 응용)."""
import pickle, math, json, os
from datetime import datetime
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("agent-project-tools")

# KB 로드 (40 노트북에서 만든 인덱스)
with open("./outputs/agent/agent_kb.pkl", "rb") as f:
    _kb = pickle.load(f)
_docs, _vec, _mat = _kb["docs"], _kb["vectorizer"], _kb["doc_matrix"]


@mcp.tool()
def search_kb(query: str, top_k: int = 3) -> list:
    """프로젝트 지식 베이스에서 query 와 가장 유사한 문서 Top-K 를 반환한다.

    Args:
        query: 검색어 (자연어)
        top_k: 반환할 문서 수
    """
    from sklearn.metrics.pairwise import cosine_similarity
    q = _vec.transform([query])
    sims = cosine_similarity(q, _mat).ravel()
    top = sims.argsort()[::-1][:top_k]
    return [{"score": float(sims[i]), "doc": _docs[i]} for i in top]


@mcp.tool()
def calculate(expression: str) -> dict:
    """안전한 수식 계산기. + - * / ** ( ) sqrt log sin cos 만 허용한다."""
    safe = {"sqrt": math.sqrt, "log": math.log,
            "sin": math.sin, "cos": math.cos, "pi": math.pi}
    try:
        result = eval(expression, {"__builtins__": {}}, safe)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


@mcp.tool()
def get_time(timezone: str = "Asia/Seoul") -> dict:
    """현재 시각을 반환한다."""
    return {"timezone": timezone, "now": datetime.now().isoformat()}


if __name__ == "__main__":
    mcp.run(transport="stdio")


### 서버 코드 해설

- `@mcp.tool()` — 타입 힌트 + docstring 이 그대로 LLM 에 노출되는 «인터페이스 카드»가 됨
- `transport="stdio"` — 로컬 프로세스로 실행. Claude Desktop / Cursor 에서 연결 가능
- 원격 노출이 필요하면 `transport="sse"` 로 변경 + FastAPI 마운트


---

## 4️⃣ MCP 클라이언트 + LLM 브리지

세션 36 의 «진짜 에이전트 = LLM ↔ MCP Client ↔ MCP Server» 패턴을 그대로 구현.
강의 노트북에서는 외부 프로세스 띄우기 부담을 피해, **같은 도구를 in-process 함수로** 노출해
LangGraph 가 직접 호출하도록 합니다 (운영 환경에서는 stdio_client 로 교체).


In [ ]:
# 🔧 도구 함수 (in-process — MCP 서버와 동일 시그니처)

def tool_search_kb(query: str, top_k: int = 3) -> dict:
    hits = search_kb(query, top_k=top_k)
    return {"hits": hits, "n": len(hits)}


def tool_calculate(expression: str) -> dict:
    safe = {"sqrt": math.sqrt, "log": math.log,
            "sin": math.sin, "cos": math.cos, "pi": math.pi}
    try:
        result = eval(expression, {"__builtins__": {}}, safe)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


def tool_get_time(timezone: str = "Asia/Seoul") -> dict:
    return {"timezone": timezone, "now": datetime.now().isoformat()}


TOOLS = {
    "search_kb": tool_search_kb,
    "calculate": tool_calculate,
    "get_time":  tool_get_time,
}

# 도구 메타데이터 (LLM Function Calling 스펙과 호환)
TOOL_SCHEMAS = [
    {"name": "search_kb",
     "description": "프로젝트 지식 베이스에서 query 와 유사한 문서 Top-K 검색",
     "parameters": {"type": "object",
                    "properties": {"query": {"type": "string"},
                                   "top_k": {"type": "integer", "default": 3}},
                    "required": ["query"]}},
    {"name": "calculate",
     "description": "안전한 수식 계산기 (+, -, *, /, **, sqrt, log)",
     "parameters": {"type": "object",
                    "properties": {"expression": {"type": "string"}},
                    "required": ["expression"]}},
    {"name": "get_time",
     "description": "현재 시각 반환 (기본 Asia/Seoul)",
     "parameters": {"type": "object",
                    "properties": {"timezone": {"type": "string"}}}},
]

print(f"✅ in-process 도구 {len(TOOLS)}개 등록")
for name, fn in TOOLS.items():
    print(f"  - {name}({list(fn.__code__.co_varnames[:fn.__code__.co_argcount])})")


---

## 5️⃣ LangGraph 워크플로 — 라우터 + 전문 에이전트

세션 38 의 멀티 에이전트 패턴을 적용합니다.

```
            ┌─────────┐
질문 ──────►│ Router  │── 분류 ──► [kb | calc | time | direct]
            └─────────┘
                 │
        ┌────────┼────────┐
        ▼        ▼        ▼
   [kb agent] [calc] [time]
        │        │        │
        └────────┼────────┘
                 ▼
            ┌─────────┐
            │ Answer  │── LLM ──► 최종 답변
            └─────────┘
```


In [ ]:
# 🌐 라우터 — 키워드 기반 (튜토리얼용. 운영은 LLM classifier 권장)

def route(query: str) -> str:
    """질문을 보고 어떤 도구를 쓸지 결정한다."""
    q = query.lower()
    if any(kw in q for kw in ["계산", "더해", "곱해", "+", "-", "*", "/", "sqrt", "값은"]):
        return "calculate"
    if any(kw in q for kw in ["몇 시", "현재 시각", "지금 시간", "오늘"]):
        return "get_time"
    return "search_kb"   # 기본은 KB 검색


# State 정의 (LangGraph 가 있으면 그대로, 없으면 dict 로 폴백)
class AgentState(TypedDict, total=False):
    query: str
    route: str
    tool_result: dict
    answer: str


def node_route(state: AgentState) -> AgentState:
    state["route"] = route(state["query"])
    return state


def node_search_kb(state: AgentState) -> AgentState:
    state["tool_result"] = tool_search_kb(state["query"], top_k=3)
    return state


def node_calculate(state: AgentState) -> AgentState:
    # 질문에서 수식 추출 — 한국어/구두점 제거 후 남는 부분
    expr = re.sub(r"[가-힣?!,]", " ", state["query"])
    expr = re.sub(r"\s+", "", expr).rstrip(".")
    state["tool_result"] = tool_calculate(expr or state["query"])
    return state


def node_get_time(state: AgentState) -> AgentState:
    state["tool_result"] = tool_get_time()
    return state


def node_answer(state: AgentState) -> AgentState:
    """도구 결과 + LLM 으로 최종 답변 생성."""
    tr = state.get("tool_result", {})
    if HAS_OPENAI and os.getenv("OPENAI_API_KEY"):
        try:
            client = OpenAI()
            prompt = (
                f"질문: {state['query']}\n\n"
                f"사용한 도구: {state['route']}\n"
                f"도구 결과: {json.dumps(tr, ensure_ascii=False)}\n\n"
                f"위 도구 결과를 바탕으로 자연스러운 한국어로 답하세요."
            )
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            state["answer"] = resp.choices[0].message.content
            return state
        except Exception as e:
            print(f"⚠ LLM 호출 실패 ({type(e).__name__}) — 템플릿 답변 사용")
    # 폴백 — 도구 결과 기반 템플릿 답변
    if state["route"] == "search_kb":
        hits = tr.get("hits", [])
        if hits:
            state["answer"] = f"[KB 검색] {hits[0]['doc']}"
        else:
            state["answer"] = "관련 정보를 찾지 못했습니다."
    elif state["route"] == "calculate":
        if "result" in tr:
            state["answer"] = f"[계산] {tr['expression']} = {tr['result']}"
        else:
            state["answer"] = f"[계산 오류] {tr.get('error', 'unknown')}"
    elif state["route"] == "get_time":
        state["answer"] = f"[현재 시각] {tr.get('now', '?')}"
    return state


# === LangGraph 빌드 ===
if HAS_LANGGRAPH:
    g = StateGraph(AgentState)
    g.add_node("route",     node_route)
    g.add_node("search_kb", node_search_kb)
    g.add_node("calculate", node_calculate)
    g.add_node("get_time",  node_get_time)
    g.add_node("answer",    node_answer)

    g.set_entry_point("route")
    g.add_conditional_edges("route", lambda s: s["route"],
                            {"search_kb": "search_kb",
                             "calculate": "calculate",
                             "get_time":  "get_time"})
    g.add_edge("search_kb", "answer")
    g.add_edge("calculate", "answer")
    g.add_edge("get_time",  "answer")
    g.add_edge("answer",    END)

    workflow = g.compile()
    print("✅ LangGraph 워크플로 컴파일 완료")
else:
    # 폴백 — 같은 흐름을 순차 실행하는 함수
    def workflow_invoke(state):
        state = node_route(state)
        state = {"search_kb": node_search_kb,
                 "calculate": node_calculate,
                 "get_time":  node_get_time}[state["route"]](state)
        state = node_answer(state)
        return state
    class _W:
        def invoke(self, s): return workflow_invoke(s)
    workflow = _W()
    print("✅ 폴백 워크플로 사용 (LangGraph 없음)")


---

## 6️⃣ A2A 인터페이스 — Agent Card + JSON-RPC 엔드포인트 (Session 37 응용)

이 에이전트를 «외부에서 다른 에이전트가 호출할 수 있도록» A2A 프로토콜로 노출합니다.
세션 37 의 패턴 그대로 — `/.well-known/agent.json` 명함 + `/a2a` JSON-RPC 엔드포인트.


In [ ]:
%%writefile a2a_server.py
"""A2A 인터페이스 — 본 에이전트를 다른 에이전트가 호출하도록 노출 (Session 37 응용)."""
from fastapi import FastAPI
from pydantic import BaseModel
import uuid, json
from agent_pipeline import AgentPipeline   # 다음 셀에서 만듦

app = FastAPI(title="Agent Project — A2A endpoint")
agent = AgentPipeline()


@app.get("/.well-known/agent.json")
async def agent_card():
    \"\"\"Agent Card — 이 에이전트의 명함.\"\"\"
    return {
        "name": "agent-project-knowledge",
        "description": "프로젝트 지식 베이스 + 계산 + 시각 조회 에이전트",
        "url": "http://localhost:8003/a2a",
        "capabilities": {"streaming": False, "pushNotifications": False},
        "skills": [
            {"id": "kb_search",  "name": "지식 검색",
             "inputModes": ["text"], "outputModes": ["text"]},
            {"id": "calculate",  "name": "사칙연산/수학",
             "inputModes": ["text"], "outputModes": ["text"]},
            {"id": "get_time",   "name": "현재 시각",
             "inputModes": ["text"], "outputModes": ["text"]},
        ],
    }


class JsonRpc(BaseModel):
    jsonrpc: str = "2.0"
    id: str
    method: str
    params: dict


@app.post("/a2a")
async def handle(req: JsonRpc):
    \"\"\"JSON-RPC handler — message/send · tasks/get.\"\"\"
    if req.method == "message/send":
        text = req.params["message"]["parts"][0]["text"]
        result = agent.run(text)
        return {"jsonrpc": "2.0", "id": req.id,
                "result": {"id": uuid.uuid4().hex, "status": "completed",
                           "artifact": {"text": result["answer"],
                                        "route": result["route"]}}}
    return {"jsonrpc": "2.0", "id": req.id,
            "error": {"code": -32601, "message": "Method not found"}}


# 실행: uvicorn a2a_server:app --port 8003


> **실행 방법**:
> 1. 다음 셀의 `agent_pipeline.py` 까지 저장 후
> 2. 터미널에서 `uvicorn a2a_server:app --port 8003`
> 3. 다른 에이전트가 `GET http://localhost:8003/.well-known/agent.json` 으로 Discovery
> 4. `POST http://localhost:8003/a2a` 로 `message/send` 호출


---

## 7️⃣ AgentPipeline — 41 평가 / 42 배포가 import 할 모듈

위에서 구현한 라우터 + 도구 + LangGraph 흐름을 **재사용 가능한 클래스** 로 묶어 파일로 저장합니다.


In [ ]:
%%writefile agent_pipeline.py
"""AgentPipeline — KB + Tools + LangGraph 통합 에이전트.
41(평가)·42(배포) 노트북에서 그대로 임포트해 사용한다."""
import os, pickle, math, re, json
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity


class AgentPipeline:
    def __init__(self, kb_path: str = "./outputs/agent/agent_kb.pkl"):
        with open(kb_path, "rb") as f:
            kb = pickle.load(f)
        self.docs = kb["docs"]
        self.vec = kb["vectorizer"]
        self.mat = kb["doc_matrix"]

    # ─── 도구 ───
    def search_kb(self, query: str, top_k: int = 3) -> dict:
        q = self.vec.transform([query])
        sims = cosine_similarity(q, self.mat).ravel()
        top = sims.argsort()[::-1][:top_k]
        return {"hits": [{"score": float(sims[i]), "doc": self.docs[i]}
                         for i in top]}

    def calculate(self, expression: str) -> dict:
        safe = {"sqrt": math.sqrt, "log": math.log,
                "sin": math.sin, "cos": math.cos, "pi": math.pi}
        try:
            return {"expression": expression,
                    "result": eval(expression, {"__builtins__": {}}, safe)}
        except Exception as e:
            return {"expression": expression, "error": str(e)}

    def get_time(self, timezone: str = "Asia/Seoul") -> dict:
        return {"timezone": timezone, "now": datetime.now().isoformat()}

    # ─── 라우터 ───
    def route(self, query: str) -> str:
        q = query.lower()
        if any(k in q for k in ["계산", "더해", "곱해", "+", "-", "*", "/", "sqrt"]):
            return "calculate"
        if any(k in q for k in ["몇 시", "현재 시각", "지금 시간"]):
            return "get_time"
        return "search_kb"

    # ─── 답변 생성 ───
    def _llm_answer(self, query, route, tr):
        try:
            from openai import OpenAI
            client = OpenAI()
            prompt = (f"질문: {query}\n"
                      f"도구: {route}\n"
                      f"도구 결과: {json.dumps(tr, ensure_ascii=False)}\n\n"
                      f"위 결과를 바탕으로 한국어로 답하세요.")
            return client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            ).choices[0].message.content
        except Exception:
            return None

    # ─── 메인 실행 ───
    def run(self, query: str) -> dict:
        r = self.route(query)
        # 수식 추출 — 한국어/구두점 제거
        expr = re.sub(r"[가-힣?!,]", " ", query)
        expr = re.sub(r"\s+", "", expr).rstrip(".") or query
        tr = {"search_kb": lambda: self.search_kb(query),
              "calculate": lambda: self.calculate(expr),
              "get_time":  lambda: self.get_time()}[r]()

        ans = self._llm_answer(query, r, tr)
        if ans is None:
            if r == "search_kb" and tr["hits"]:
                ans = f"[KB] {tr['hits'][0]['doc']}"
            elif r == "calculate" and "result" in tr:
                ans = f"[계산] {tr['expression']} = {tr['result']}"
            elif r == "get_time":
                ans = f"[현재 시각] {tr['now']}"
            else:
                ans = "답변을 생성하지 못했습니다."
        return {"query": query, "route": r, "tool_result": tr, "answer": ans}


---

## 8️⃣ 통합 테스트 — 다양한 질의로 라우팅·답변 검증


In [ ]:
# 🧪 통합 테스트 — workflow 직접 호출

test_queries = [
    "MCP 가 뭔지 설명해줘",            # → KB
    "12 + 7 * 3 의 값은?",             # → 계산
    "지금 시간 몇 시야?",               # → 시각
    "A2A 프로토콜의 특징은?",          # → KB
    "sqrt(64) 의 값은?",                # → 계산
]

print("=" * 70)
for q in test_queries:
    print(f"\nQ: {q}")
    result = workflow.invoke({"query": q})
    print(f"  ↳ route: {result.get('route')}")
    print(f"  ↳ answer: {result.get('answer', '')[:150]}")
print("\n" + "=" * 70)
print(f"\n✅ {len(test_queries)}개 질의 처리 완료")


---

## 9️⃣ 산출물 저장 — 다음 노트북(41/42)에서 사용


In [ ]:
# 💾 agent_config.json 저장

agent_config = {
    "agent_name":   "agent-project-knowledge",
    "kb_path":      os.path.join(OUTPUT_DIR, "agent_kb.pkl"),
    "mcp_server":   "mcp_server.py",
    "a2a_server":   "a2a_server.py",
    "pipeline":     "agent_pipeline.py",
    "tools":        list(TOOLS.keys()),
    "n_docs":       len(docs),
    "uses_langgraph": HAS_LANGGRAPH,
    "uses_openai":  HAS_OPENAI and bool(os.getenv("OPENAI_API_KEY")),
}

config_path = os.path.join(OUTPUT_DIR, "agent_config.json")
json.dump(agent_config, open(config_path, "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

print(f"✅ 저장: {config_path}")
for k, v in agent_config.items():
    print(f"  {k:18s} = {v}")


---

## 📝 정리

### 이번 노트북에서 만든 것

| 구성 | 사용 기술 | 세션 |
|---|---|---|
| 지식 베이스 | TF-IDF 인덱스 | (39 데이터) |
| 도구 서버 | FastMCP — `search_kb · calculate · get_time` | **Session 36** |
| 워크플로 | LangGraph (Router → Specialist → Answer) | **Session 38** |
| 외부 노출 | A2A — Agent Card + `/a2a` JSON-RPC | **Session 37** |
| 재사용 모듈 | `AgentPipeline` 클래스 | (Tool Calling 일반) |

### 핵심 메시지

> 세션 35~38 에서 따로 배운 4개 표준이 «하나의 진짜 프로젝트» 안에서 어떻게 합쳐지는지 본 셈.
> MCP = 도구 노출 / LangGraph = 흐름 / A2A = 다른 에이전트와 통신 / AgentPipeline = 우리 코드의 진입점.

### 산출물 (`outputs/agent/` 디렉터리)

- `agent_kb.pkl` — KB 인덱스
- `agent_config.json` — 에이전트 메타데이터
- `mcp_server.py` — FastMCP 도구 서버 (별도 실행)
- `a2a_server.py` — A2A FastAPI 노출 (별도 실행)
- `agent_pipeline.py` — 41/42 가 임포트할 모듈

### 다음 노트북

- **Session 41**: `AgentPipeline.run()` 의 응답 품질을 BLEU + LLM-as-a-Judge 로 평가
- **Session 42**: `AgentPipeline` 을 FastAPI + Streamlit 으로 감싸 배포
